## Orchestrator-Workers Workflow
In this workflow, a central LLM dynamically breaks down tasks, delegates them to worker LLMs, and synthesizes their results.

### When to use this workflow
This workflow is well-suited for complex tasks where you can't predict the subtasks needed. The key difference from simple parallelization is its flexibility—subtasks aren't pre-defined, but determined by the orchestrator based on the specific input.

### Improvements from [original cookbook notebook](https://github.com/anthropics/anthropic-cookbook/blob/main/patterns/agents/orchestrator_workers.ipynb)

- The `context` variable for the task is unused so its been removed (delete unused code).
- Original prompt tended to lead the model too much (prompt engineering).
- Original doesn't actually synthesize the results after using the workers, added it in (meet spec).

### Usage Notes
- Needs better error handling when structure isn't met. claude-3-5-haiku couldn't keep to XML spec at all.


In [1]:

# load required gems and add some helpers for pretty printing in iruby
require_relative "../../../../notebook" 

require "rexml/document"
def extract_xml(text, path)
    xml = REXML::Document.new("<root>#{text}</root>") # Wrapping in <root> to make valid XML
    xml.elements["root/"+ path].text.strip
end

# set the default model
Instruct.set_default_model "claude-3-5-sonnet-latest", access_token: ENV['ANTHROPIC_API_KEY']

true

In [2]:
# def parse_tasks(tasks_xml) -> List[Dict]:
#     """Parse XML tasks into a list of task dictionaries."""
#     tasks = []
#     current_task = {}
    
#     for line in tasks_xml.split('\n'):
#         line = line.strip()
#         if not line:
#             continue
            
#         if line.startswith("<task>"):
#             current_task = {}
#         elif line.startswith("<type>"):
#             current_task["type"] = line[6:-7].strip()
#         elif line.startswith("<description>"):
#             current_task["description"] = line[12:-13].strip()
#         elif line.startswith("</task>"):
#             if "description" in current_task:
#                 if "type" not in current_task:
#                     current_task["type"] = "default"
#                 tasks.append(current_task
    
#     return tasks

# class FlexibleOrchestrator:
#     """Break down tasks and run them in parallel using worker LLMs."""
    
#     def __init__(
#         self,
#         orchestrator_prompt: str,
#         worker_prompt: str,
#     ):
#         """Initialize with prompt templates."""
#         self.orchestrator_prompt = orchestrator_prompt
#         self.worker_prompt = worker_prompt

#     def _format_prompt(self, template: str, **kwargs) -> str:
#         """Format a prompt template with variables."""
#         try:
#             return template.format(**kwargs)
#         except KeyError as e:
#             raise ValueError(f"Missing required prompt variable: {e}")

#     def process(self, task: str, context: Optional[Dict] = None) -> Dict:
#         """Process task by breaking it down and running subtasks in parallel."""
#         context = context or {}
        
#         # Step 1: Get orchestrator response
#         orchestrator_input = self._format_prompt(
#             self.orchestrator_prompt,
#             task=task,
#             **context
#         )
#         orchestrator_response = llm_call(orchestrator_input)
        
#         # Parse orchestrator response
#         analysis = extract_xml(orchestrator_response, "analysis")
#         tasks_xml = extract_xml(orchestrator_response, "tasks")
#         tasks = parse_tasks(tasks_xml)
        
#         print("\n=== ORCHESTRATOR OUTPUT ===")
#         print(f"\nANALYSIS:\n{analysis}")
#         print(f"\nTASKS:\n{tasks}")
        
#         # Step 2: Process each task
#         worker_results = []
#         for task_info in tasks:
#             worker_input = self._format_prompt(
#                 self.worker_prompt,
#                 original_task=task,
#                 task_type=task_info['type'],
#                 task_description=task_info['description'],
#                 **context
#             )
            
#             worker_response = llm_call(worker_input)
#             result = extract_xml(worker_response, "response")
            
#             worker_results.append({
#                 "type": task_info["type"],
#                 "description": task_info["description"],
#                 "result": result
#             })
            
#             print(f"\n=== WORKER RESULT ({task_info['type']}) ===\n{result}\n")
        
#         return {
#             "analysis": analysis,
#             "worker_results": worker_results,
#         }

In [3]:
task="Write a product description for a new eco-friendly water bottle"

orchestrator_prompt = p.user{"
Analyze this task and break it down into 2-3 distinct approaches:

Task: <%= task %>

Return your response in this format:

<analysis>
Explain your understanding of the task and which variations would be valuable.
Focus on how each approach serves different aspects of the task.
</analysis>

<tasks>
    <task>
        <style>[writing style]</style>
        <guidelines>[writing guidelines]</guidelines>
    </task>
</tasks>
"} + gen

result = orchestrator_prompt.call(temperature: 0.1)

{ source: llm, stop_reason: end_turn, stream_chunk: 3 }<analysis>
This task requires creating marketing{ stream_chunk: 4 } copy for a water bottle with environmental{ stream_chunk: 5 } benefits. Key variations would be valuable base{ stream_chunk: 6 }d on:
1. Emotional{ stream_chunk: 7 } vs. technical focus
2. Length{ stream_chunk: 8 } and detail level
3. Primary selling point{ stream_chunk: 9 } (environmental impact vs. practical benefits{ stream_chunk: 10 })

Different approaches serve distinct purposes:{ stream_chunk: 11 }
- A technical approach builds credibility and appeals{ stream_chunk: 12 } to detail-oriented consumers
- An emotional/{ stream_chunk: 13 }lifestyle approach connects with environmentally conscious consumers
- A balance{ stream_chunk: 14 }d approach combines practical benefits with environmental impact
</analysis>

{ stream_chunk: 15 }<tasks>
    <task>
        { stream_chunk: 16 }<style>Technical and Specification-{ stream_chunk: 17 }Focused</style>
        <guidelines>{ stream_chunk: 18 }
        - Lead with material specifications and eco-credentials{ stream_chunk: 19 }
        - Include specific metrics (e.g.,{ stream_chunk: 20 } plastic bottles saved)
        - Focus on durability and construction{ stream_chunk: 21 } details
        - Use precise language about{ stream_chunk: 22 } environmental impact
        - Highlight technical{ stream_chunk: 23 } innovations in design
        </guidelines>{ stream_chunk: 24 }
    </task>

    <task>{ stream_chunk: 25 }
        <style>Lifestyle and Emotional Impact{ stream_chunk: 26 }</style>
        <guidelines>
        - Open{ stream_chunk: 27 } with environmental mission statement
        - Tell a story about environmental{ stream_chunk: 28 } impact
        - Use evocative, inspiring{ stream_chunk: 29 } language
        - Focus on personal contribution to sustainability
        -{ stream_chunk: 30 } Include aspirational lifestyle elements
        </guidelines>
    </task>{ stream_chunk: 31 }

    <task>
        <style>{ stream_chunk: 32 }Practical Benefits with Eco-Context</style>
        { stream_chunk: 33 }<guidelines>
        - Lead with everyday usage{ stream_chunk: 34 } benefits
        - Weave in environmental benefits{ stream_chunk: 35 } as supporting points
        - Balance features{ stream_chunk: 36 } with sustainability impact
        - Use accessible{ stream_chunk: 37 }, clear language
        - Include{ stream_chunk: 38 } both practical and eco-friendly features
        </guidelines>{ stream_chunk: 39 }
    </task>
</tasks>{ -source, -stop_reason, -stream_chunk }

In [8]:
xml =  REXML::Document.new("<root>#{result}</root>")
puts "Analysis:\n#{xml.elements['root/analysis'].text.strip}"
worker_tasks = []
REXML::XPath.each(xml,'root/tasks/task') do |task|
    worker_tasks << { style: task.elements['style'].text.strip, guidelines: task.elements['guidelines'].text.strip }
end

worker_prompts = worker_tasks.map do |worker_task|
    p.user{"Generate content based on:
Task: <%= task %>
Style: <%= worker_task[:style] %>
Guidelines: <%= worker_task[:guidelines] %>

Return your response in this format:

<response>
Your content here, maintaining the specified style and fully addressing requirements.
</response>
"} + gen
end

worker_results = worker_prompts.map(&:call).map{ |result| extract_xml(result, "response") }

Analysis:
This task requires creating marketing copy for a water bottle with environmental benefits. Key variations would be valuable based on:
1. Emotional vs. technical focus
2. Length and detail level
3. Primary selling point (environmental impact vs. practical benefits)

Different approaches serve distinct purposes:
- A technical approach builds credibility and appeals to detail-oriented consumers
- An emotional/lifestyle approach connects with environmentally conscious consumers
- A balanced approach combines practical benefits with environmental impact


["The EcoTech Pro X30 Sustainable Hydration Vessel\n\nEngineered from aerospace-grade 316L stainless steel with proprietary ThermaShield™ technology, the EcoTech Pro X30 represents the pinnacle of sustainable hydration engineering. This 30oz (887ml) vessel features a triple-wall vacuum-sealed construction, achieving 97.8% thermal retention efficiency over 24 hours.\n\nTechnical Specifications:\n- Material: 316L marine-grade stainless steel (100% recyclable)\n- Wall thickness: 1.2mm\n- Impact resistance: Tested to withstand drops from 2.1m\n- Thermal performance: Hot 12hrs (±2°C) / Cold 24hrs (±1°C)\n- Weight: 340g\n- Capacity: 887ml (30oz)\n\nEnvironmental Impact Metrics:\n- Eliminates 167 single-use plastic bottles annually (based on average daily use)\n- Carbon footprint offset within 31 days of regular use\n- 100% plastic-free packaging using biodegradable kraft materials\n- Production facility operates on 100% renewable energy\n\nInnovation Features:\n- Proprietary BioCote™ antimic

In [18]:
orchestrator_prompt = p.user{"
Take the best of these 2-3 distinct approaches to the same writing task and decide on the best to publish:

Task: <%= task %>
<% worker_results.each do |result| %>
-----
 <%= result %>
<% end %>

Return your response in this format:

<thoughts>
Your thoughts before answer
</thoughts>
<rewritten>
[rewritten as well as possible]
</rewritten>
"} + gen
result = orchestrator_prompt.call
puts extract_xml(result,"rewritten")

Transform Your Daily Hydration with the EcoFlow Premium Sustainable Water Bottle

Stay perfectly hydrated while making a real environmental impact with the EcoFlow, a premium 24oz water bottle that combines innovative design with environmental responsibility. Engineered from high-grade recycled stainless steel, this durable vessel keeps your beverages cold for 24 hours or hot for 12 hours, making it your perfect companion from morning coffee to evening refreshments.

Designed for real-life convenience, the EcoFlow features:
- Advanced leak-proof technology with one-handed opening mechanism
- Double-wall vacuum insulation for superior temperature control
- Wide-mouth design for easy filling and cleaning
- Ergonomic grip for comfortable handling
- Removable strainer for fruit infusions or tea
- Measurement markers to track daily hydration

Environmental Impact:
- Prevents 167 single-use plastic bottles from landfills annually
- Made from 100% recyclable materials
- Zero-waste packaging f